In [1]:
#Following document is for preprocessing. 

Following document is for preprocessing


!pip uninstall -y edgar edgartools
!pip install -U edgartools
from edgar import *

In [2]:
!pip install lxml_html_clean
!pip install newspaper
!pip install newspaper3k
!pip install lxml_html_clean
!pip install dataclasses
!pip install spacy

  Using cached newspaper-0.1.0.7.tar.gz (176 kB)
  Preparing metadata (setup.py) ... error
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [1 lines of output]
      WARNING! You are attempting to install newspaper's python2 repository on python3. PLEASE RUN `$ pip3 install newspaper3k` for python3 or `$ pip install newspaper` for python2
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [3]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 12.3 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


#need News api below 

#Step for getting the newsAPI

In [ ]:
#code for getting the news api
#next one
# ============================================================
# Hybrid News Retrieval Pipeline — Google Colab
# v8 — Interactive inputs · Expanded free sources
#      3-stage scraper · Subject-aware filtering
#      Boilerplate-aware first paragraph cleaning
# ============================================================
# Cell 1 — install dependencies (run once):
#
#   !pip install newspaper3k feedparser requests python-dateutil spacy -q
#   !python -m spacy download en_core_web_sm -q
#
# Cell 2 — paste and run this entire cell
# ============================================================

import re
import logging
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass

import requests
import feedparser
import spacy
from dateutil import parser as dateutil_parser
from newspaper import Article

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ============================================================
# ★  YOUR NEWSAPI KEY
# ============================================================
NEWSAPI_KEY    = ""    # paste your key here
MAX_PER_SOURCE = 7
# ============================================================

# ── Interactive inputs ───────────────────────────────────────────
print("=" * 60)
print("  HYBRID NEWS RETRIEVAL PIPELINE")
print("=" * 60)
company_name = input("\n  Enter company name (e.g. Meta, Google, Apple): ").strip()
days_input   = input("  Enter days to look back — max 25 (press Enter for 25): ").strip()
DAYS_BACK    = int(days_input) if days_input.isdigit() else 25
DAYS_BACK    = min(DAYS_BACK, 25)  # enforce free tier limit
QUESTION     = f"What is happening with {company_name} lately?"
print(f"\n  Company  : {company_name}")
print(f"  Days back: {DAYS_BACK}")
print(f"  Question : {QUESTION}")
print("=" * 60 + "\n")

NEWSAPI_URL  = "https://newsapi.org/v2/everything"
MAX_WORKERS  = 10
MIN_TEXT_LEN = 150
PAGE_SIZE    = 100

# ── Browser-like headers to avoid scraping blocks ───────────────
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate",
    "Connection":      "keep-alive",
}

# ── Free sources only ────────────────────────────────────────────
NEWSAPI_DOMAINS = ",".join([
    "reuters.com",
    "apnews.com",
    "cnbc.com",
    "marketwatch.com",
    "finance.yahoo.com",
    "thestreet.com",
    "nasdaq.com",
    "fortune.com",
    "fool.com",
    "businesswire.com",
    "prnewswire.com",
    "techcrunch.com",
    "venturebeat.com",
    "wired.com",
    "arstechnica.com",
    "zdnet.com",
    "theguardian.com",
])

# ── Source priority ──────────────────────────────────────────────
SOURCE_PRIORITY = {
    "reuters":          10,
    "associated press": 10,
    "ap news":          10,
    "cnbc":             9,
    "marketwatch":      9,
    "yahoo finance":    8,
    "nasdaq":           8,
    "fortune":          8,
    "the street":       7,
    "thestreet":        7,
    "the guardian":     7,
    "business wire":    7,
    "businesswire":     7,
    "pr newswire":      7,
    "prnewswire":       7,
    "techcrunch":       6,
    "venturebeat":      6,
    "wired":            6,
    "ars technica":     6,
    "arstechnica":      6,
    "zdnet":            5,
    "motley fool":      5,
}

# ── RSS feeds ────────────────────────────────────────────────────
RSS_FEEDS = [
    ("Reuters Business",      "https://feeds.reuters.com/reuters/businessNews"),
    ("Reuters Technology",    "https://feeds.reuters.com/reuters/technologyNews"),
    ("AP News Business",      "https://feeds.apnews.com/rss/apf-business"),
    ("CNBC Top News",         "https://feeds.content.dowjones.io/public/rss/mktgNewsHeadlines"),
    ("CNBC Technology",       "https://feeds.content.dowjones.io/public/rss/mktgTechNewsHeadlines"),
    ("MarketWatch",           "https://feeds.marketwatch.com/marketwatch/topstories"),
    ("The Motley Fool",       "https://www.fool.com/feeds/index.aspx"),
    ("The Street",            "https://www.thestreet.com/rss/index.xml"),
    ("TechCrunch",            "https://techcrunch.com/category/startups/feed/"),
    ("VentureBeat",           "https://venturebeat.com/feed/"),
    ("Wired Business",        "https://www.wired.com/feed/category/business/latest/rss"),
    ("Ars Technica",          "https://feeds.arstechnica.com/arstechnica/index"),
 #   ("ZDNet",                 "https://www.zdnet.com/news/rss.xml"),
    ("PR Newswire Tech",      "https://www.prnewswire.com/rss/news-releases-list.rss"),
    ("Business Wire",         "https://feed.businesswire.com/rss/home/?rss=G1&_gl=1"),
    ("The Guardian Business", "https://www.theguardian.com/business/rss"),
    ("The Guardian Tech",     "https://www.theguardian.com/technology/rss"),
    ("BBC Business",          "https://feeds.bbci.co.uk/news/business/rss.xml"),
    ("Fortune",               "https://fortune.com/feed"),
]

# ── Company aliases ──────────────────────────────────────────────
ALIAS_MAP = {
    "meta":       {"meta platforms", "facebook"},
    "apple":      {"aapl"},
    "google":     {"alphabet", "googl"},
    "amazon":     {"amzn"},
    "tesla":      {"tsla"},
    "nvidia":     {"nvda"},
    "microsoft":  {"msft"},
    "netflix":    {"nflx"},
    "uber":       {"uber technologies"},
    "twitter":    {"x corp", "x.com"},
    "snapchat":   {"snap inc", "snap"},
    "spotify":    {"spot"},
    "airbnb":     {"abnb"},
    "lyft":       {"lyft inc"},
    "samsung":    {"samsung electronics"},
    "intel":      {"intc"},
    "amd":        {"advanced micro devices"},
    "salesforce": {"crm"},
    "oracle":     {"orcl"},
    "ibm":        {"international business machines"},
}

# ── Boilerplate patterns for body text filter ────────────────────
BOILERPLATE_PATTERNS = [
    r"subscribe to our newsletter",
    r"sign up for",
    r"enable javascript",
    r"cookies? to improve",
    r"please enable",
    r"advertisement",
]

# ── Navigation/promo lines to strip before subject check ─────────
HEADER_NOISE_PATTERNS = [
    r"add .+ as your preferred",
    r"^share$",
    r"^read more$",
    r"^advertisement$",
    r"^skip to",
    r"^sign in$",
    r"^subscribe$",
    r"^follow us",
    r"^newsletter$",
]

# ─── data model ─────────────────────────────────────────────────
@dataclass
class CandidateArticle:
    title:        str
    source:       str
    published_at: datetime
    url:          str
    raw_text:     str = ""
    drop_reason:  str = ""

# ─── 1. NER — detect company ────────────────────────────────────
def detect_company(question: str) -> str:
    nlp  = spacy.load("en_core_web_sm")
    doc  = nlp(question)
    orgs = [ent.text for ent in doc.ents if ent.label_ == "ORG"]
    if orgs:
        log.info(f"NER detected company: {orgs[0]}")
        return orgs[0]
    skip = {"what","who","how","when","where","why","is","are","the","a","an"}
    for token in doc:
        if token.text[0].isupper() and token.text.lower() not in skip:
            log.info(f"NER fallback company: {token.text}")
            return token.text
    return ""

# ─── 2. Temporal NER ────────────────────────────────────────────
def detect_days_back(question: str) -> int:
    q = question.lower()
    patterns = [
        (r"\btoday\b|\bright now\b|\bcurrently\b",  1),
        (r"\bthis week\b|\blately\b|\brecently\b",  7),
        (r"\blast week\b",                           7),
        (r"\bthis month\b|\blast month\b",          30),
        (r"\bthis year\b|\blast year\b",            365),
        (r"\blast (\d+) days?\b",                   "days"),
        (r"\blast (\d+) weeks?\b",                  "weeks"),
        (r"\blast (\d+) months?\b",                 "months"),
    ]
    for pattern, unit in patterns:
        m = re.search(pattern, q)
        if m:
            if isinstance(unit, int):
                log.info(f"Temporal NER: '{m.group()}' → {unit} days")
                return unit
            n      = int(m.group(1))
            result = n * (7 if unit=="weeks" else 30 if unit=="months" else 1)
            log.info(f"Temporal NER: '{m.group()}' → {result} days")
            return result
    log.info("Temporal NER: no match, defaulting to 7 days")
    return 7

# ─── 3a. NewsAPI retrieval ───────────────────────────────────────
def fetch_newsapi(company: str, days_back: int) -> list:
    if not NEWSAPI_KEY or NEWSAPI_KEY == "YOUR_NEWSAPI_KEY_HERE":
        log.warning("NewsAPI key not set — skipping NewsAPI. Only RSS will be used.")
        return []

    from_date = (
        datetime.now(timezone.utc) - timedelta(days=days_back)
    ).strftime("%Y-%m-%d")

    all_articles = []
    seen_urls    = set()

    # ── Call 1: main domain list ─────────────────────────────────
    params_main = {
        "q":        f'"{company}"',
        "from":     from_date,
        "sortBy":   "relevancy",
        "pageSize": PAGE_SIZE,
        "language": "en",
        "domains":  NEWSAPI_DOMAINS,
        "apiKey":   NEWSAPI_KEY,
    }
    try:
        resp = requests.get(NEWSAPI_URL, params=params_main, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        if data.get("status") == "ok":
            for item in data.get("articles", []):
                url = item.get("url", "")
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    try:
                        pub = dateutil_parser.parse(
                            item["publishedAt"]
                        ).replace(tzinfo=timezone.utc)
                    except Exception:
                        pub = datetime.now(timezone.utc)
                    all_articles.append(CandidateArticle(
                        title=item.get("title", ""),
                        source=item.get("source", {}).get("name", "NewsAPI"),
                        published_at=pub,
                        url=url,
                    ))
            log.info(
                f"NewsAPI main: {len(all_articles)} candidates "
                f"(total available: {data.get('totalResults','?')})"
            )
        else:
            log.error(f"NewsAPI main error: {data.get('message', 'unknown')}")
    except requests.RequestException as e:
        log.error(f"NewsAPI main request failed: {e}")

    # ── Call 2: Yahoo Finance specifically ───────────────────────
    params_yahoo = {
        "q":        f'"{company}" site:finance.yahoo.com',
        "from":     from_date,
        "sortBy":   "relevancy",
        "pageSize": 20,
        "language": "en",
        "apiKey":   NEWSAPI_KEY,
    }
    try:
        resp_yf = requests.get(NEWSAPI_URL, params=params_yahoo, timeout=10)
        resp_yf.raise_for_status()
        data_yf = resp_yf.json()
        yf_count = 0
        if data_yf.get("status") == "ok":
            for item in data_yf.get("articles", []):
                url = item.get("url", "")
                if url and url not in seen_urls and "yahoo" in url.lower():
                    seen_urls.add(url)
                    yf_count += 1
                    try:
                        pub = dateutil_parser.parse(
                            item["publishedAt"]
                        ).replace(tzinfo=timezone.utc)
                    except Exception:
                        pub = datetime.now(timezone.utc)
                    all_articles.append(CandidateArticle(
                        title=item.get("title", ""),
                        source="Yahoo Finance",
                        published_at=pub,
                        url=url,
                    ))
            log.info(f"NewsAPI Yahoo Finance: {yf_count} additional candidates")
        else:
            log.warning(
                f"NewsAPI Yahoo Finance: {data_yf.get('message', 'unknown')}"
            )
    except requests.RequestException as e:
        log.error(f"NewsAPI Yahoo Finance request failed: {e}")

    log.info(f"NewsAPI total: {len(all_articles)} candidates")
    return all_articles

# ─── 3b. RSS retrieval ───────────────────────────────────────────
def fetch_rss_feed(
    feed_name: str, feed_url: str, company: str, days_back: int
) -> list:
    cutoff        = datetime.now(timezone.utc) - timedelta(days=days_back)
    company_lower = company.lower()
    articles      = []
    try:
        feed = feedparser.parse(feed_url)
    except Exception as e:
        log.warning(f"RSS fetch failed for {feed_name}: {e}")
        return []

    for entry in feed.entries:
        try:
            pub_struct = (
                entry.get("published_parsed") or entry.get("updated_parsed")
            )
            pub = (
                datetime(*pub_struct[:6], tzinfo=timezone.utc)
                if pub_struct
                else datetime.now(timezone.utc)
            )
        except Exception:
            pub = datetime.now(timezone.utc)

        if pub < cutoff:
            continue

        title   = entry.get("title", "")
        summary = entry.get("summary", "")
        url     = entry.get("link", "")

        if company_lower not in (title + " " + summary).lower():
            continue

        articles.append(CandidateArticle(
            title=title, source=feed_name,
            published_at=pub, url=url,
        ))

    log.info(f"RSS '{feed_name}': {len(articles)} matches for '{company}'")
    return articles

def fetch_all_rss(company: str, days_back: int) -> list:
    all_articles = []
    with ThreadPoolExecutor(max_workers=len(RSS_FEEDS)) as executor:
        futures = {
            executor.submit(fetch_rss_feed, name, url, company, days_back): name
            for name, url in RSS_FEEDS
        }
        for future in as_completed(futures):
            try:
                all_articles.extend(future.result())
            except Exception as e:
                log.warning(f"RSS future error: {e}")
    return all_articles

# ─── 4. Merge · deduplicate · prioritise · cap ──────────────────
def get_source_priority(source: str) -> int:
    source_lower = source.lower()
    return max(
        (v for k, v in SOURCE_PRIORITY.items() if k in source_lower),
        default=3
    )

def merge_and_deduplicate(newsapi_articles: list, rss_articles: list) -> list:
    seen_urls = set()
    merged    = []

    for art in newsapi_articles:
        if art.url and art.url not in seen_urls:
            seen_urls.add(art.url)
            merged.append(art)
    for art in rss_articles:
        if art.url and art.url not in seen_urls:
            seen_urls.add(art.url)
            merged.append(art)

    merged.sort(
        key=lambda a: (get_source_priority(a.source), a.published_at),
        reverse=True
    )

    source_counts = defaultdict(int)
    capped        = []
    for art in merged:
        if source_counts[art.source] < MAX_PER_SOURCE:
            capped.append(art)
            source_counts[art.source] += 1

    log.info(
        f"Merged: {len(newsapi_articles)} NewsAPI + {len(rss_articles)} RSS "
        f"→ {len(merged)} unique → {len(capped)} after source cap"
    )
    counts = Counter(art.source for art in capped)
    for src, n in counts.most_common():
        log.info(f"  {src:<35} : {n} articles")
    return capped

# ─── 5. Improved 3-stage scraper ────────────────────────────────
def scrape_article(art: CandidateArticle) -> CandidateArticle:
    """
    Stage 1: newspaper3k default downloader
    Stage 2: requests with browser User-Agent → re-parse with newspaper3k
    Stage 3: raw <p> tag extraction as final fallback
    """
    try:
        # Stage 1
        a = Article(art.url)
        a.download()
        a.parse()
        art.raw_text = a.text.strip()

        # Stage 2
        if not art.raw_text or len(art.raw_text) < MIN_TEXT_LEN:
            resp = requests.get(art.url, headers=HEADERS, timeout=12)
            if resp.status_code == 200:
                a2 = Article(art.url)
                a2.set_html(resp.text)
                a2.parse()
                candidate = a2.text.strip()
                if len(candidate) > len(art.raw_text):
                    art.raw_text = candidate

        # Stage 3
        if not art.raw_text or len(art.raw_text) < MIN_TEXT_LEN:
            try:
                resp = requests.get(art.url, headers=HEADERS, timeout=12)
                if resp.status_code == 200:
                    paragraphs  = re.findall(
                        r'<p[^>]*>(.*?)</p>', resp.text,
                        re.DOTALL | re.IGNORECASE
                    )
                    clean_paras = [
                        re.sub(r'<[^>]+>', '', p).strip()
                        for p in paragraphs
                    ]
                    body = "\n\n".join(p for p in clean_paras if len(p) > 40)
                    if len(body) > len(art.raw_text):
                        art.raw_text = body
            except Exception:
                pass

    except Exception as e:
        if not art.raw_text:
            art.drop_reason = f"scrape_error: {e}"

    return art

def parallel_scrape(candidates: list) -> list:
    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(scrape_article, art): art for art in candidates
        }
        for future in as_completed(futures):
            try:
                results.append(future.result())
            except Exception as e:
                log.warning(f"Scrape future error: {e}")
    return results

# ─── 6. First paragraph cleaner ─────────────────────────────────
def clean_first_para(text: str) -> str:
    """
    Strip navigation and promotional lines from the top of
    article text before subject check. This prevents boilerplate
    like 'Add AP News on Google' from being treated as a
    subject mention of Google.
    """
    lines = text.split('\n')
    clean = [
        line for line in lines
        if not any(
            re.search(pattern, line.strip().lower())
            for pattern in HEADER_NOISE_PATTERNS
        )
    ]
    return '\n'.join(clean)[:500].lower()

# ─── 7. Heuristic + subject-aware filtering ─────────────────────
def apply_filters(articles: list, company: str):
    company_lower = company.lower()
    aliases       = {company_lower} | ALIAS_MAP.get(company_lower, set())
    kept, dropped = [], []

    for art in articles:

        # ── already flagged by scraper ───────────────────────────
        if art.drop_reason:
            dropped.append(art); continue

        # ── empty text ───────────────────────────────────────────
        if not art.raw_text.strip():
            art.drop_reason = "empty_text"
            dropped.append(art); continue

        # ── too short ────────────────────────────────────────────
        if len(art.raw_text) < MIN_TEXT_LEN:
            art.drop_reason = f"too_short ({len(art.raw_text)} chars)"
            dropped.append(art); continue

        # ── boilerplate body ─────────────────────────────────────
        if any(re.search(p, art.raw_text.lower()) for p in BOILERPLATE_PATTERNS):
            art.drop_reason = "boilerplate"
            dropped.append(art); continue

        # ── subject check ────────────────────────────────────────
        # company must appear in title OR cleaned first paragraph.
        # clean_first_para() strips nav/promo lines first so that
        # boilerplate like "Add AP News on Google" does not count
        # as a subject match for Google.
        title_lower   = art.title.lower()
        first_para    = clean_first_para(art.raw_text)
        in_title      = any(alias in title_lower  for alias in aliases)
        in_first_para = any(alias in first_para   for alias in aliases)

        if not (in_title or in_first_para):
            art.drop_reason = "not_subject (not in title or first para)"
            dropped.append(art); continue

        kept.append(art)

    log.info(f"Filtering: {len(kept)} kept, {len(dropped)} dropped")
    return kept, dropped

# ─── 8. Content verification ────────────────────────────────────
def verify_content(df: list, df_articles: list):
    print(f"\n{'='*60}")
    print(f"  CONTENT VERIFICATION")
    print(f"{'='*60}")

    if not df:
        print("  No kept articles to verify.")
        return

    total_chars = sum(len(a.raw_text) for a in df)
    avg_chars   = total_chars // len(df)
    print(f"  Kept articles      : {len(df)}")
    print(f"  Total content      : {total_chars:,} chars")
    print(f"  Average per article: {avg_chars:,} chars")

    counts = Counter(a.source for a in df)
    print(f"\n  Source breakdown:")
    for src, n in counts.most_common():
        print(f"    {src:<35} : {n} articles")
    print(f"{'='*60}\n")

    for i, art in enumerate(df, 1):
        char_count = len(art.raw_text)
        print(f"  ── Article {i} {'─'*44}")
        print(f"  Title  : {art.title}")
        print(f"  Source : {art.source}")
        print(f"  Date   : {art.published_at.strftime('%Y-%m-%d %H:%M')}")
        print(f"  URL    : {art.url[:70]}...")
        print(f"  Length : {char_count:,} chars")
        print(f"\n  First 500 chars of raw_text:")
        print(f"  {'-'*50}")
        preview = art.raw_text[:500].replace('\n', '\n  ')
        print(f"  {preview}")
        if char_count > 500:
            print(f"  ... [{char_count - 500:,} more chars]")
        print()

    if df_articles:
        print(f"\n{'='*60}")
        print(f"  DROPPED — REASON BREAKDOWN")
        print(f"{'='*60}")
        reasons = Counter(
            a.drop_reason.split("(")[0].strip()
            for a in df_articles if a.drop_reason
        )
        for reason, count in reasons.most_common():
            print(f"  {reason:<40} : {count} articles")
        print()

# ─── 9. Main pipeline ────────────────────────────────────────────
def run_pipeline(question: str):
    print(f"\n{'='*60}")
    print(f"  QUESTION : {question}")
    print(f"{'='*60}\n")

    company = detect_company(question)
    if not company:
        print("ERROR: Could not extract company name from question.")
        return None, None

    days_back = DAYS_BACK if DAYS_BACK else detect_days_back(question)

    print(f"  Company       : {company}")
    print(f"  Days back     : {days_back}")
    print(f"  Max per source: {MAX_PER_SOURCE}")
    print(f"  NewsAPI domains: {len(NEWSAPI_DOMAINS.split(','))}")
    print(f"  RSS feeds     : {len(RSS_FEEDS)}\n")

    with ThreadPoolExecutor(max_workers=2) as executor:
        f_newsapi        = executor.submit(fetch_newsapi, company, days_back)
        f_rss            = executor.submit(fetch_all_rss,  company, days_back)
        newsapi_articles = f_newsapi.result()
        rss_articles     = f_rss.result()

    candidates = merge_and_deduplicate(newsapi_articles, rss_articles)
    if not candidates:
        print("No candidate articles found.")
        return [], []

    print(f"\n  Scraping {len(candidates)} URLs (3-stage scraper)...\n")
    scraped       = parallel_scrape(candidates)
    kept, dropped = apply_filters(scraped, company)

    print(f"\n{'='*60}")
    print(f"  PIPELINE RESULTS — '{company}' ({days_back} days)")
    print(f"{'='*60}")
    print(f"  NewsAPI candidates  : {len(newsapi_articles)}")
    print(f"  RSS candidates      : {len(rss_articles)}")
    print(f"  After dedup + cap   : {len(candidates)}")
    print(f"  Kept after filtering: {len(kept)}")
    print(f"  Dropped             : {len(dropped)}")
    print(f"{'='*60}")

    if kept:
        print("\n  KEPT ARTICLES:")
        for i, art in enumerate(kept, 1):
            priority = get_source_priority(art.source)
            print(f"  {i}. [P{priority}] [{art.source}] {art.title[:55]}...")
            print(f"     {art.published_at.strftime('%Y-%m-%d')} | {len(art.raw_text):,} chars")
            print()

    if dropped:
        print("\n  DROPPED (debug):")
        for art in dropped:
            print(f"  - [{art.drop_reason:<40}] {art.title[:40]}...")

    print()
    verify_content(kept, dropped)
    return kept, dropped

# ─── 10. Convert to dataframe ────────────────────────────────────
def to_dataframe(kept: list, dropped: list):
    import pandas as pd

    df_final = pd.DataFrame([{
        "company":      company_name,
        "title":        art.title,
        "source":       art.source,
        "published_at": art.published_at,
        "url":          art.url,
        "raw_text":     art.raw_text,
        "char_count":   len(art.raw_text),
        "priority":     get_source_priority(art.source),
    } for art in kept])

    df_debug = pd.DataFrame([{
        "company":      company_name,
        "title":        art.title,
        "source":       art.source,
        "published_at": art.published_at,
        "url":          art.url,
        "drop_reason":  art.drop_reason,
    } for art in dropped])

    print(f"\n  df_final : {len(df_final)} rows × {len(df_final.columns)} cols")
    print(f"  df_debug : {len(df_debug)} rows × {len(df_debug.columns)} cols\n")
    return df_final, df_debug

# ─── 11. Multi-query runner (optional) ───────────────────────────
def run_multi_query(base_company: str, extra_terms: list = None):
    """
    Run pipeline for multiple related queries and combine results.
    Useful for getting more training data volume.

    Example:
        df_combined, _ = run_multi_query("Meta", ["Meta Platforms", "Mark Zuckerberg"])
    """
    queries     = [base_company] + (extra_terms or [])
    all_kept    = []
    all_dropped = []

    for term in queries:
        print(f"\n{'─'*60}")
        print(f"  Running query for: {term}")
        print(f"{'─'*60}")
        k, d = run_pipeline(f"What is happening with {term} lately?")
        if k: all_kept.extend(k)
        if d: all_dropped.extend(d)

    seen        = set()
    unique_kept = []
    for art in all_kept:
        if art.url not in seen:
            seen.add(art.url)
            unique_kept.append(art)

    print(f"\n{'='*60}")
    print(f"  MULTI-QUERY SUMMARY")
    print(f"{'='*60}")
    print(f"  Queries run     : {len(queries)}")
    print(f"  Total kept      : {len(all_kept)}")
    print(f"  After URL dedup : {len(unique_kept)}")
    print(f"{'='*60}\n")

    return to_dataframe(unique_kept, all_dropped)

# ─── RUN ─────────────────────────────────────────────────────────
kept, dropped      = run_pipeline(QUESTION)
df_final, df_debug = to_dataframe(kept, dropped)
from IPython.display import display

print("  df_final preview:")
print(df_final[["company", "title", "source", "published_at", "char_count", "priority"]].to_string())

# ── Optional: run multiple related queries for more volume ───────
# Uncomment to use:
#
# df_combined, df_debug_combined = run_multi_query(
#     company_name,
#     extra_terms=[f"{company_name} stock", f"{company_name} earnings"]
# )
# display(df_combined[["company", "title", "source", "published_at", "char_count"]])

  HYBRID NEWS RETRIEVAL PIPELINE

  Company  : Meta
  Days back: 25
  Question : What is happening with Meta lately?


  QUESTION : What is happening with Meta lately?



22:11:56  INFO      NER fallback company: Meta


  Company       : Meta
  Days back     : 25
  Max per source: 7
  NewsAPI domains: 17
  RSS feeds     : 18



22:11:57  INFO      NewsAPI main: 100 candidates (total available: 190)
22:11:57  INFO      NewsAPI Yahoo Finance: 0 additional candidates
22:11:57  INFO      NewsAPI total: 100 candidates
22:11:58  INFO      RSS 'AP News Business': 0 matches for 'Meta'
22:11:58  INFO      RSS 'Reuters Business': 0 matches for 'Meta'
22:11:58  INFO      RSS 'Reuters Technology': 0 matches for 'Meta'
22:11:58  INFO      RSS 'The Street': 0 matches for 'Meta'
22:11:58  INFO      RSS 'TechCrunch': 0 matches for 'Meta'
22:11:58  INFO      RSS 'Ars Technica': 1 matches for 'Meta'
22:11:58  INFO      RSS 'Wired Business': 1 matches for 'Meta'
22:11:58  INFO      RSS 'CNBC Technology': 0 matches for 'Meta'
22:11:58  INFO      RSS 'Fortune': 0 matches for 'Meta'
22:11:58  INFO      RSS 'PR Newswire Tech': 0 matches for 'Meta'
22:11:58  INFO      RSS 'MarketWatch': 1 matches for 'Meta'
22:11:58  INFO      RSS 'BBC Business': 1 matches for 'Meta'
22:11:58  INFO      RSS 'The Guardian Business': 1 matches for 'Me


  Scraping 41 URLs (3-stage scraper)...



22:12:02  INFO      Filtering: 13 kept, 28 dropped



  PIPELINE RESULTS — 'Meta' (25 days)
  NewsAPI candidates  : 100
  RSS candidates      : 5
  After dedup + cap   : 41
  Kept after filtering: 13
  Dropped             : 28

  KEPT ARTICLES:
  1. [P8] [Fortune] David’s Bridal exec has a warning for every CEO obsesse...
     2026-04-22 | 4,031 chars

  2. [P9] [CNBC] Meta is tracking employee keystrokes on Google, LinkedI...
     2026-04-23 | 3,853 chars

  3. [P7] [The Guardian Business] Kenyan firm sacks more than 1,000 workers after losing ...
     2026-04-17 | 2,686 chars

  4. [P8] [Fortune] Meta will start tracking employees’ screens and keystro...
     2026-04-21 | 2,311 chars

  5. [P6] [Wired] New Gas-Powered Data Centers Could Emit More Greenhouse...
     2026-04-22 | 3,297 chars

  6. [P6] [Ars Technica] Greenhouse gases from data center boom could outpace en...
     2026-04-23 | 14,909 chars

  7. [P6] [Wired] Meta’s New AI Asked for My Raw Health Data—and Gave Me ...
     2026-04-10 | 2,976 chars

  8. [P6] [Wired] Meta Is

In [45]:
news_final = df_final

In [46]:
news_final = df_final


In [47]:
news_final # final news into a dataframe 

,company,title,source,published_at,url,raw_text,char_count,priority
0,Meta,David’s Bridal exec has a warning for every CE...,Fortune,2026-04-22 19:10:30+00:00,https://fortune.com/2026/04/22/davids-bridal-e...,After years working in marketing positions at ...,4031,8
1,Meta,Meta is tracking employee keystrokes on Google...,CNBC,2026-04-23 00:18:18+00:00,https://www.cnbc.com/2026/04/22/meta-tracks-em...,"Mark Zuckerberg, chief executive officer of Me...",3853,9
2,Meta,"Kenyan firm sacks more than 1,000 workers afte...",The Guardian Business,2026-04-17 16:59:04+00:00,https://www.theguardian.com/technology/2026/ap...,"More than 1,000 low-paid workers in Kenya have...",2686,7
3,Meta,Meta will start tracking employees’ screens an...,Fortune,2026-04-21 18:53:46+00:00,https://fortune.com/2026/04/21/meta-will-start...,Meta is installing tracking software on U.S. e...,2311,8
4,Meta,New Gas-Powered Data Centers Could Emit More G...,Wired,2026-04-22 10:30:00+00:00,https://www.wired.com/story/new-gas-powered-da...,New gas projects linked to just 11 data center...,3297,6
5,Meta,Greenhouse gases from data center boom could o...,Ars Technica,2026-04-23 14:51:16+00:00,https://arstechnica.com/ai/2026/04/greenhouse-...,New gas projects linked to just 11 data center...,14909,6
6,Meta,Meta’s New AI Asked for My Raw Health Data—and...,Wired,2026-04-10 09:30:00+00:00,https://www.wired.com/story/metas-new-ai-asked...,Meta’s Superintelligence Labs launched its fir...,2976,6
7,Meta,Meta Is Warned That Facial Recognition Glasses...,Wired,2026-04-13 16:01:30+00:00,https://www.wired.com/story/meta-ray-ban-oakle...,"More than 70 civil liberties, domestic violenc...",4366,6
8,Meta,"Best Meta Glasses (2026): Ray-Ban, Oakley, AR",Wired,2026-04-19 11:00:00+00:00,https://www.wired.com/story/best-meta-glasses/,Every time I’ve written about Meta’s AI-enable...,2817,6
9,Meta,"Meta says it will cut 8,000 jobs as AI spendin...",BBC Business,2026-04-23 22:56:00+00:00,https://www.bbc.com/news/articles/crm1y89vek8o...,A key reason for the layoffs is Meta's increas...,289,3


#For Getting past 4 quarters of earning report:
#please change the company_name and ticker to extract past 4 earning reports. 
#please also change the set_identity email to your email. this is just guideline from SEC that they want to know which email is retrieving the data. s

In [11]:
import pandas as pd
from edgar import Company, set_identity

# ── Set identity (required by SEC) ───────────────────────────────
set_identity("spso771522@gmail.com")

# ============================================================
# ★  CONFIG
# ============================================================
TICKER       = "AAPL"       # stock ticker
COMPANY_NAME = "Apple"  # company name
MAX_EARNINGS = 4            # number of quarters to pull
MAX_SCAN     = 30           # max 8-K filings to scan through
# ============================================================

# ─── 1. Extract text from filing object ──────────────────────────
def extract_text_from_filing(filing_obj) -> str:
    """Try multiple methods to get text from a filing object."""

    # method 1 — press release text (best for 8-K earnings)
    try:
        prs = filing_obj.press_releases
        if prs and len(prs) > 0:
            pr = prs[0]
            for attr in ["text", "markdown", "content", "html"]:
                if hasattr(pr, attr):
                    val = getattr(pr, attr)
                    if callable(val):
                        try:
                            result = val()
                            if isinstance(result, str) and len(result) > 200:
                                return result[:50000]
                        except Exception:
                            pass
                    elif isinstance(val, str) and len(val) > 200:
                        return val[:50000]
    except Exception:
        pass

    # method 2 — filing .text() method
    try:
        text = filing_obj.text()
        if isinstance(text, str) and len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    # method 3 — filing .markdown() method
    try:
        text = filing_obj.markdown()
        if isinstance(text, str) and len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    # method 4 — str conversion fallback
    try:
        text = str(filing_obj)
        if len(text) > 200:
            return text[:50000]
    except Exception:
        pass

    return ""

# ─── 2. Check if 8-K is an earnings filing ───────────────────────
def is_earnings_filing(filing_obj) -> bool:
    """Check if this 8-K contains earnings results (Item 2.02)."""

    # check items list
    try:
        items = filing_obj.items
        if any("2.02" in str(item) for item in items):
            return True
    except Exception:
        pass

    # check string representation
    try:
        filing_str = str(filing_obj).lower()
        if any(kw in filing_str for kw in [
            "2.02", "results of operations",
            "quarterly results", "financial results",
            "earnings per share", "revenue was"
        ]):
            return True
    except Exception:
        pass

    return False

# ─── 3. Pull 4 quarters of earnings 8-K ──────────────────────────
def get_earnings_filings(ticker: str) -> pd.DataFrame:
    """
    Pull last MAX_EARNINGS earnings 8-K press releases.
    Takes full press release text for proper chunking.
    Returns df_edgar DataFrame.
    """
    print(f"\n{'='*60}")
    print(f"  EDGAR EARNINGS COLLECTION — {ticker}")
    print(f"  Target: last {MAX_EARNINGS} earnings quarters")
    print(f"{'='*60}")

    # get company
    print(f"\n  Loading {ticker}...")
    company = Company(ticker)
    print(f"  ✓ Found: {company.name}")

    # get 8-K filings
    print(f"\n  Fetching 8-K filings (scanning up to {MAX_SCAN})...")
    filings = company.get_filings(form="8-K").head(MAX_SCAN)

    results = []

    for filing in filings:
        try:
            filing_date = str(filing.filing_date)
            obj         = filing.obj()

            # check if earnings
            if not is_earnings_filing(obj):
                continue

            # get full text
            text = extract_text_from_filing(obj)

            if not text or len(text.strip()) < 200:
                print(f"  ✗ No text found  : {filing_date}")
                continue

            results.append({
                "ticker":       ticker.upper(),
                "company":      COMPANY_NAME,
                "section_type": "8-K earnings",
                "filing_type":  "8-K",
                "filing_date":  filing_date,
                "text":         text.strip(),
            })

            print(f"  ✓ {filing_date} : {len(text):,} chars")

            if len(results) >= MAX_EARNINGS:
                print(f"\n  ✓ Collected {MAX_EARNINGS} quarters")
                break

        except Exception as e:
            print(f"  Skipping {filing.filing_date}: {e}")

    # ── Summary ───────────────────────────────────────────────────
    if not results:
        print(f"\n  ✗ No earnings 8-K found for {ticker}")
        return pd.DataFrame()

    df_edgar = pd.DataFrame(results)

    print(f"\n{'='*60}")
    print(f"  EARNINGS COLLECTION SUMMARY")
    print(f"{'='*60}")
    print(f"  Quarters collected : {len(df_edgar)}")
    print(f"\n  By filing date:")
    for _, row in df_edgar.iterrows():
        print(f"    {row['filing_date']} : {len(row['text']):,} chars")
    print(f"{'='*60}\n")

    return df_edgar

# ============================================================
# ★  RUN
# ============================================================

df_edgar = get_earnings_filings(TICKER)

if len(df_edgar) > 0:
    print(f"✓ df_edgar ready : {len(df_edgar)} earnings quarters")
    print(f"\n  Preview:")
    print(df_edgar[["ticker", "company", "filing_date",
                    "filing_type"]].to_string())
    print(f"\n  Text lengths:")
    for _, row in df_edgar.iterrows():
        print(f"    {row['filing_date']} : {len(row['text']):,} chars")
    print(f"\n  NEXT STEP: run chunking cell")
else:
    print(f"✗ No earnings data collected for {TICKER}")

21:52:35  INFO      Identity of the Edgar REST client set to [spso771522@gmail.com]



  EDGAR EARNINGS COLLECTION — AAPL
  Target: last 4 earnings quarters

  Loading AAPL...
  ✓ Found: Apple Inc.

  Fetching 8-K filings (scanning up to 30)...


21:52:35  INFO      Pattern detection successful: 1 sections found
21:52:35  INFO      Pattern detection successful: 2 sections found
21:52:36  INFO      Pattern detection successful: 2 sections found


  ✓ 2026-01-29 : 24,031 chars


21:52:36  INFO      Pattern detection successful: 1 sections found
21:52:36  INFO      Pattern detection successful: 1 sections found
21:52:37  INFO      Pattern detection successful: 2 sections found
21:52:37  INFO      Pattern detection successful: 2 sections found


  ✓ 2025-10-30 : 31,524 chars
  ✓ 2025-07-31 : 25,826 chars


21:52:38  INFO      Pattern detection successful: 1 sections found
21:52:38  INFO      Pattern detection successful: 1 sections found
21:52:38  INFO      Pattern detection successful: 2 sections found
21:52:39  INFO      Pattern detection successful: 2 sections found


  ✓ 2025-05-01 : 26,119 chars

  ✓ Collected 4 quarters

  EARNINGS COLLECTION SUMMARY
  Quarters collected : 4

  By filing date:
    2026-01-29 : 24,023 chars
    2025-10-30 : 31,521 chars
    2025-07-31 : 25,820 chars
    2025-05-01 : 26,112 chars

✓ df_edgar ready : 4 earnings quarters

  Preview:
  ticker company filing_date filing_type
0   AAPL   Apple  2026-01-29         8-K
1   AAPL   Apple  2025-10-30         8-K
2   AAPL   Apple  2025-07-31         8-K
3   AAPL   Apple  2025-05-01         8-K

  Text lengths:
    2026-01-29 : 24,023 chars
    2025-10-30 : 31,521 chars
    2025-07-31 : 25,820 chars
    2025-05-01 : 26,112 chars

  NEXT STEP: run chunking cell


In [12]:
#four elements as inputs. to be combined. 
df_edgar


,ticker,company,section_type,filing_type,filing_date,text
0,AAPL,Apple,8-K earnings,8-K,2026-01-29,Document\n\nExhibit 99.1\nApple reports first ...
1,AAPL,Apple,8-K earnings,8-K,2025-10-30,Document\n\nExhibit 99.1\nApple reports fourth...
2,AAPL,Apple,8-K earnings,8-K,2025-07-31,Document\n\nExhibit 99.1\nApple reports third ...
3,AAPL,Apple,8-K earnings,8-K,2025-05-01,Document\n\nExhibit 99.1\nApple reports second...


In [13]:
#lets take out the risk for now? 
mda_text

NameError: name 'mda_text' is not defined

In [14]:
tenk.risk_factors

NameError: name 'tenk' is not defined

In [15]:
news_final

,company,title,source,published_at,url,raw_text,char_count,priority
0,Apple,Steve Jobs called Tim Cook ‘not a product pers...,Fortune,2026-04-22 13:42:03+00:00,https://fortune.com/2026/04/22/tim-cook-retiri...,Tim Cook and Steve Jobs couldn’t have been mor...,3723,8
1,Apple,"Here’s what Warren Buffett, Sam Altman, Donald...",Fortune,2026-04-21 16:35:02+00:00,https://fortune.com/2026/04/21/warren-buffett-...,Apple is entering a new era. The company annou...,4296,8
2,Apple,Tim Cook is exceptional at this leadership ski...,CNBC,2026-04-22 12:55:32+00:00,https://www.cnbc.com/2026/04/22/tim-cook-is-ex...,"Apple CEO Tim Cook speaks during Apple's ""Awe-...",6087,9
3,Apple,"John Ternus, the man stepping into Tim Cook an...",Fortune,2026-04-21 16:26:54+00:00,https://fortune.com/2026/04/21/who-is-john-ter...,Apple’s next CEO John Ternus is a company vete...,4633,8
4,Apple,Apple is slipping on Tim Cook’s exit. Wall Str...,Fortune,2026-04-21 15:57:16+00:00,https://fortune.com/2026/04/21/apple-stock-sli...,Tim Cook is handing Apple to its hardware chie...,3140,8
5,Apple,New Apple CEO John Ternus doubted himself when...,CNBC,2026-04-21 16:12:31+00:00,https://www.cnbc.com/2026/04/21/john-ternus-ap...,Apple is adding a new CEO to its ranks and con...,1477,9
6,Apple,Tim Cook turned Apple into a $4 trillion jugge...,CNBC,2026-04-21 15:14:19+00:00,https://www.cnbc.com/2026/04/21/tim-cook-apple...,In this article AAPL Follow your favorite stoc...,8996,9
7,Apple,Apple just named its next CEO—and Tim Cook is ...,Fortune,2026-04-21 15:14:13+00:00,https://fortune.com/2026/04/21/apple-next-ceo-...,"Apple just named its next CEO, who will be tak...",4651,8
8,Apple,A new generation is reviving the iPod for dist...,Associated Press,2026-04-13 07:05:46+00:00,https://apnews.com/article/ipod-music-streamin...,Add AP News as your preferred source to see mo...,7525,10
9,Apple,"Who is John Ternus, Apple’s next CEO?",The Guardian Tech,2026-04-21 01:48:37+00:00,https://www.theguardian.com/technology/2026/ap...,Apple has announced longtime company veteran J...,3142,7


In [16]:
mda_text

NameError: name 'mda_text' is not defined

In [ ]:
#getting risk data

In [17]:
import os
from edgar import set_identity, Company

# Set identity (still required!)
set_identity("Sang Park spso771522@gmail.com")
# Get company financials
financials = Company("MSFT").get_financials()
#First version of NLP:
#getting the risk factors from 10-K April 18th
from edgar import Company
company = Company("MSFT")            # or any ticker
filing = company.get_filings(form="10-K").latest()
tenk = filing.obj()
risk_text = tenk.risk_factors
print(risk_text)
risk_text
#First step: getting the second risk factor (Item 1)
from edgar import Company
company = Company("MSFT")
filings = company.get_filings(form="10-K").head(3)
for filing in filings:
    tenk = filing.obj()
    print("=" * 80)
    print(f"{filing.company} | {filing.filing_date} | {filing.form}")
    print(tenk.risk_factors[:3000])   # first 3000 chars
#the U.S. Securities and Exchange Commission (SEC), specifically:
#Form 10-K → annual report (most comprehensive filing)

21:53:02  INFO      Identity of the Edgar REST client set to [Sang Park spso771522@gmail.com]
21:53:11  INFO      TOC detection found 23 sections
21:53:11  INFO      TOC detection successful: 23 sections found


ITEM 1A. RISK FACTORS

Our operations and financial results are subject to various risks and uncertainties, including those described below, that could adversely affect our business, operations, financial condition, results of operations, liquidity, and the trading price of our common stock.

STRATEGIC AND COMPETITIVE RISKS

We face intense competition across all markets for our products and services, which could adversely affect our results of operations.

Competition in the technology sector

Our competitors range in size from diversified global companies with significant research and development resources to small, specialized firms whose narrower product lines may let them be more effective in deploying technical, marketing, and financial resources. Barriers to entry in many of our businesses are low and many of the areas in which we compete evolve rapidly with changing and disruptive technologies, shifting user needs, and frequent introductions of new products and services. If we 

21:53:20  INFO      TOC detection found 23 sections
21:53:20  INFO      TOC detection successful: 23 sections found


ITEM 1A. RISK FACTORS

Our operations and financial results are subject to various risks and uncertainties, including those described below, that could adversely affect our business, operations, financial condition, results of operations, liquidity, and the trading price of our common stock.

STRATEGIC AND COMPETITIVE RISKS

We face intense competition across all markets for our products and services, which could adversely affect our results of operations.

Competition in the technology sector

Our competitors range in size from diversified global companies with significant research and development resources to small, specialized firms whose narrower product lines may let them be more effective in deploying technical, marketing, and financial resources. Barriers to entry in many of our businesses are low and many of the areas in which we compete evolve rapidly with changing and disruptive technologies, shifting user needs, and frequent introductions of new products and services. If we 

21:53:28  INFO      TOC detection found 23 sections
21:53:28  INFO      TOC detection successful: 23 sections found


ITEM 1A. RISK FACTORS

  Our operations and financial results are subject to various risks and uncertainties, including those described below, that could adversely affect our business, operations, financial condition, results of operations, liquidity, and the trading price of our common stock.

  STRATEGIC AND COMPETITIVE RISKS

  We face intense competition across all markets for our products and services, which may adversely affect our results of operations.

  Competition in the technology sector

  Our competitors range in size from diversified global companies with significant research and development resources to small, specialized firms whose narrower product lines may let them be more effective in deploying technical, marketing, and financial resources. Barriers to entry in many of our businesses are low and many of the areas in which we compete evolve rapidly with changing and disruptive technologies, shifting user needs, and frequent introductions of new products and services

21:53:37  INFO      TOC detection found 22 sections
21:53:37  INFO      TOC detection successful: 22 sections found


ITEM 1A. RISK FACTORS

  Our operations and financial results are subject to various risks and uncertainties, including those described below, that could adversely affect our business, financial condition, results of operations, cash flows, and the trading price of our common stock.

  STRATEGIC AND COMPETITIVE RISKS

  We face intense competition across all markets for our products and services, which may lead to lower revenue or operating margins.

  Competition in the technology sector

  Our competitors range in size from diversified global companies with significant research and development resources to small, specialized firms whose narrower product lines may let them be more effective in deploying technical, marketing, and financial resources. Barriers to entry in many of our businesses are low and many of the areas in which we compete evolve rapidly with changing and disruptive technologies, shifting user needs, and frequent introductions of new products and services. Our abili

In [ ]:
# MDA text


In [18]:
#Item 7: MDA doc.
from edgar import Company, set_identity
import re
set_identity("sup2107@columbia.com")
filing = Company("MSFT").get_filings(form="10-K")[0]
text = filing.text()
pattern = re.compile(
    r'(item\s*7\.?\s*management.*?discussion.*?analysis.*?)(item\s*7a\.|item\s*8\.)',
    re.IGNORECASE | re.DOTALL
)
candidates = []
for m in pattern.finditer(text):
    section = m.group(1).strip()
    if len(section) > 2000:   # skip tiny TOC-style matches
        candidates.append(section)
if candidates:
    mda_text = max(candidates, key=len)
    print(mda_text[:5000])
else:
    print("MD&A section not found")

21:53:37  INFO      Identity of the Edgar REST client set to [sup2107@columbia.com]


ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS

The following Management’s Discussion and Analysis of Financial Condition and Results of Operations (“MD&A”) is intended to help the reader understand the results of operations and financial condition of Microsoft Corporation. MD&A is provided as a supplement to, and should be read in conjunction with, our consolidated financial statements and the accompanying Notes to Financial Statements (Part II, Item 8 of this Form 10-K). This section generally discusses the results of our operations for the
year ended June 30, 2025 compared to the year ended June 30, 2024. For a discussion of the year ended June 30, 2024 compared to the year ended June 30, 2023, please refer to Part II, Item 7, “Management’s Discussion and Analysis of Financial Condition and Results of Operations” in our Annual Report on Form 10-K for the year ended June 30, 2024 and our Form 8-K filed on December 3, 2024.

OVERVIEW

Mic

In [19]:
mda_final =mda_text

In [ ]:
#build function for chunking. 

In [20]:

from nltk.tokenize import sent_tokenize   # import directly — avoids module clash
import re
import nltk
import pandas as pd
from dataclasses import dataclass, field

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── Chunking config ──────────────────────────────────────────────
MAX_TOKENS          = 400
OVERLAP_SENTENCES   = 1
MIN_CHUNK_CHARS     = 100
AVG_CHARS_PER_TOKEN = 4

# ── Earnings boilerplate to strip ────────────────────────────────
STRIP_PATTERNS_EARNINGS = [
    r"(?i)forward.looking statements.{0,2000}",
    r"(?i)safe harbor.{0,1000}",
    r"(?i)private securities litigation.{0,500}",
    r"(?i)(investor relations|media contact|for more information).{0,300}",
    r"(?i)^(document|exhibit\s*\d+\.\d+)\s*$",
]

# ─── data model ─────────────────────────────────────────────────
@dataclass
class Chunk:
    text:         str
    company:      str
    ticker:       str
    source:       str
    source_type:  str
    published_at: str
    url:          str
    chunk_index:  int
    chunk_total:  int
    char_count:   int = field(init=False)
    token_count:  int = field(init=False)

    def __post_init__(self):
        self.char_count  = len(self.text)
        self.token_count = len(self.text) // AVG_CHARS_PER_TOKEN

# ─── 1. Text cleaner ────────────────────────────────────────────
def clean_text(text: str, source_type: str = "news") -> str:
    if not text:
        return ""

    # strip HTML
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"&[a-z]+;", " ", text)

    if source_type == "earnings":
        # strip legal boilerplate
        for pattern in STRIP_PATTERNS_EARNINGS:
            text = re.sub(pattern, "", text, flags=re.DOTALL | re.MULTILINE)
        # strip lines that are mostly numbers (financial table rows)
        # threshold raised to 40% alphabetic
        lines       = text.split('\n')
        clean_lines = []
        for line in lines:
            stripped    = line.strip()
            alpha_chars = sum(1 for c in stripped if c.isalpha())
            total_chars = len(stripped)
            if total_chars == 0 or alpha_chars / total_chars >= 0.4:
                clean_lines.append(line)
        text = '\n'.join(clean_lines)

    # normalise whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    return text.strip()

# ─── 2. Semantic chunker ────────────────────────────────────────
def semantic_chunk(text: str, max_tokens: int = MAX_TOKENS,
                   overlap: int = OVERLAP_SENTENCES) -> list:
    try:
        sentences = sent_tokenize(text)   # using directly imported function
    except Exception:
        sentences = re.split(r'(?<=[.!?])\s+', text)

    if not sentences:
        return []

    chunks     = []
    current    = []
    cur_tokens = 0

    for sent in sentences:
        sent_tokens = len(sent) // AVG_CHARS_PER_TOKEN
        if cur_tokens + sent_tokens > max_tokens and current:
            chunks.append(' '.join(current))
            current    = current[-overlap:] if overlap > 0 else []
            cur_tokens = sum(len(s) // AVG_CHARS_PER_TOKEN for s in current)
        current.append(sent)
        cur_tokens += sent_tokens

    if current:
        chunks.append(' '.join(current))

    return chunks

# ─── 3. Fixed chunker (fallback) ────────────────────────────────
def fixed_chunk(text: str, max_tokens: int = MAX_TOKENS,
                overlap_chars: int = 50) -> list:
    max_chars = max_tokens * AVG_CHARS_PER_TOKEN
    chunks    = []
    start     = 0
    while start < len(text):
        end = start + max_chars
        chunks.append(text[start:end].strip())
        start = end - overlap_chars
    return [c for c in chunks if c]

# ─── 4. Hybrid chunker ──────────────────────────────────────────
def hybrid_chunk(text: str, max_tokens: int = MAX_TOKENS,
                 source_type: str = "news") -> list:
    text = clean_text(text, source_type)
    if not text:
        return []

    chunks       = semantic_chunk(text, max_tokens)
    final_chunks = []

    for chunk in chunks:
        if len(chunk) // AVG_CHARS_PER_TOKEN > max_tokens:
            final_chunks.extend(fixed_chunk(chunk, max_tokens))
        else:
            final_chunks.append(chunk)

    return [
        c.strip() for c in final_chunks
        if len(c.strip()) >= MIN_CHUNK_CHARS
    ]

# ─── 5. Chunk a single article ──────────────────────────────────
def chunk_article(raw_text, company, ticker, source,
                  source_type, published_at, url,
                  max_tokens=MAX_TOKENS) -> list:
    texts = hybrid_chunk(str(raw_text), max_tokens, source_type)
    if not texts:
        return []
    chunks = []
    for i, chunk_text in enumerate(texts):
        chunks.append(Chunk(
            text         = chunk_text,
            company      = company,
            ticker       = ticker.upper(),
            source       = source,
            source_type  = source_type,
            published_at = str(published_at)[:10],
            url          = str(url),
            chunk_index  = i,
            chunk_total  = len(texts),
        ))
    return chunks

# ─── 6. Chunk news df_final ─────────────────────────────────────
def chunk_news_dataframe(df, ticker, max_tokens=MAX_TOKENS) -> list:
    all_chunks = []
    for _, row in df.iterrows():
        chunks = chunk_article(
            raw_text     = row.get("raw_text", ""),
            company      = row.get("company", ""),
            ticker       = ticker,
            source       = row.get("source", ""),
            source_type  = "news",
            published_at = row.get("published_at", ""),
            url          = row.get("url", ""),
            max_tokens   = max_tokens,
        )
        all_chunks.extend(chunks)
    print(f"News chunking    : {len(df)} articles → {len(all_chunks)} chunks")
    return all_chunks

# ─── 7. Chunk earnings report ───────────────────────────────────
def chunk_earnings_report(raw_text, company, ticker,
                           published_at, max_tokens=MAX_TOKENS) -> list:
    chunks = chunk_article(
        raw_text     = raw_text,
        company      = company,
        ticker       = ticker,
        source       = "SEC EDGAR 8-K",
        source_type  = "earnings",
        published_at = published_at,
        url          = f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={ticker}",
        max_tokens   = max_tokens,
    )
    print(f"Earnings chunking: 1 report → {len(chunks)} chunks")
    return chunks

# ─── 8. Convert to dataframe ────────────────────────────────────
def chunks_to_dataframe(chunks: list) -> pd.DataFrame:
    if not chunks:
        return pd.DataFrame()
    return pd.DataFrame([{
        "company":              c.company,
        "ticker":               c.ticker,
        "source":               c.source,
        "source_type":          c.source_type,
        "published_at":         c.published_at,
        "url":                  c.url,
        "chunk_index":          c.chunk_index,
        "chunk_total":          c.chunk_total,
        "text":                 c.text,
        "char_count":           c.char_count,
        "token_count":          c.token_count,
        "sentiment_label":      None,
        "sentiment_confidence": None,
        "embedding":            None,
    } for c in chunks])

# ─── 9. Verify chunks ───────────────────────────────────────────
def verify_chunks(df_chunks: pd.DataFrame):
    print(f"\n{'='*60}")
    print(f"  CHUNK VERIFICATION")
    print(f"{'='*60}")
    print(f"  Total chunks     : {len(df_chunks)}")
    if len(df_chunks) == 0:
        print("  No chunks to verify.")
        return
    print(f"  Avg token count  : {df_chunks['token_count'].mean():.0f}")
    print(f"  Max token count  : {df_chunks['token_count'].max()}")
    print(f"  Min token count  : {df_chunks['token_count'].min()}")

    print(f"\n  By source type:")
    for stype, grp in df_chunks.groupby("source_type"):
        print(f"    {stype:<15} : {len(grp)} chunks")

    print(f"\n  By source:")
    for src, grp in df_chunks.groupby("source"):
        print(f"    {src:<35} : {len(grp)} chunks")

    print(f"\n  Sample chunks:")
    for i, row in df_chunks.head(3).iterrows():
        print(f"\n  ── Chunk {row['chunk_index']+1}/{row['chunk_total']} "
              f"[{row['source_type']}] [{row['source']}]")
        print(f"  Company : {row['company']} ({row['ticker']})")
        print(f"  Date    : {row['published_at']}")
        print(f"  Tokens  : ~{row['token_count']}")
        print(f"  Text    : {row['text'][:300]}...")
    print(f"{'='*60}\n")

# ============================================================
# ★  RUN
# ============================================================

print("=" * 60)
print("  STAGE B — HYBRID CHUNKING")
print("=" * 60)

# ── Step 1: chunk news articles (only if df_final is a valid DataFrame)
news_chunks = []
try:
    if isinstance(df_final, pd.DataFrame) and len(df_final) > 0:
        print(f"\n✓ df_final found : {len(df_final)} articles")
        # get company_name safely
        try:
            _ticker = company_name
        except NameError:
            _ticker = "UNKNOWN"
        print("Chunking news articles...")
        news_chunks = chunk_news_dataframe(
            df     = df_final,
            ticker = _ticker,
        )
    else:
        print("\n✗ df_final is empty or not a DataFrame — skipping news chunking")
except NameError:
    print("\n✗ df_final not in memory — skipping news chunking")
    print("  Run the v8 pipeline cell first to collect news articles")

# ── Step 2: chunk earnings report from text variable ─────────────
earnings_chunks = []

# ★ UPDATE THESE THREE LINES to match your company
EARNINGS_TICKER  = "MSFT"        # e.g. "META", "GOOGL", "MSFT"
EARNINGS_DATE    = "2026-01-29"  # actual filing date
EARNINGS_COMPANY = "Microsoft"       # actual company name

try:
    if text and isinstance(text, str) and len(text) > 100:
        print(f"\n✓ earnings text found : {len(text):,} chars")
        print("Chunking earnings report...")
        earnings_chunks = chunk_earnings_report(
            raw_text     = text,
            company      = EARNINGS_COMPANY,
            ticker       = EARNINGS_TICKER,
            published_at = EARNINGS_DATE,
        )
    else:
        print("\n✗ earnings text not available — skipping earnings chunking")
except NameError:
    print("\n✗ text not in memory — skipping earnings chunking")
    print("  Run get_latest_earnings_8k_text() first")

# ── Step 3: combine ──────────────────────────────────────────────
all_chunks = news_chunks + earnings_chunks
df_chunks  = chunks_to_dataframe(all_chunks)

print(f"\n{'='*60}")
print(f"  CHUNKING SUMMARY")
print(f"{'='*60}")
print(f"  News chunks      : {len(news_chunks)}")
print(f"  Earnings chunks  : {len(earnings_chunks)}")
print(f"  Total df_chunks  : {len(df_chunks)}")
print(f"{'='*60}")

# ── Step 4: verify + preview ─────────────────────────────────────
if len(df_chunks) > 0:
    verify_chunks(df_chunks)
    print("  df_chunks preview:")
    print(df_chunks[[
        "company", "ticker", "source_type", "source",
        "published_at", "chunk_index", "chunk_total",
        "token_count", "text"
    ]].head(10).to_string())
else:
    print("\n  No chunks produced.")
    print("  Make sure at least one of the following is in memory:")
    print("  1. df_final  — run v8 pipeline cell")
    print("  2. text      — run get_latest_earnings_8k_text()")

  STAGE B — HYBRID CHUNKING

✓ df_final found : 34 articles
Chunking news articles...
News chunking    : 34 articles → 126 chunks

✓ earnings text found : 349,181 chars
Chunking earnings report...
Earnings chunking: 1 report → 244 chunks

  CHUNKING SUMMARY
  News chunks      : 126
  Earnings chunks  : 244
  Total df_chunks  : 370

  CHUNK VERIFICATION
  Total chunks     : 370
  Avg token count  : 352
  Max token count  : 400
  Min token count  : 27

  By source type:
    earnings        : 244 chunks
    news            : 126 chunks

  By source:
    Ars Technica                        : 2 chunks
    Associated Press                    : 6 chunks
    CNBC                                : 31 chunks
    Fortune                             : 16 chunks
    SEC EDGAR 8-K                       : 244 chunks
    TechCrunch                          : 25 chunks
    The Guardian Business               : 3 chunks
    The Guardian Tech                   : 17 chunks
    Wired                        

Getting risk data 

getting risk data

#combine all the texts into combined dataframe. 

In [21]:
all_chunks = []
# ── 1. Chunk Risk Factors ─────────────────────────────────────────
risk_chunks = chunk_article(
    raw_text     = tenk.risk_factors,
    company      = "Microsoft",
    ticker       = "MSFT",
    source       = "SEC EDGAR 10-K",
    source_type  = "risk",
    published_at = "2026-01-28",
    url          = "https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=MSFT",
)
all_chunks.extend(risk_chunks)
print(f"✓ Risk Factors : {len(risk_chunks)} chunks")

# ── 2. Chunk MD&A ────────────────────────────────────────────────
mda_chunks = chunk_article(
    raw_text     = mda_text,
    company      = "Microsoft",
    ticker       = "MSFT",
    source       = "SEC EDGAR 10-K",
    source_type  = "mda",
    published_at = "2026-01-28",
    url          = "https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=MSFT",
)
all_chunks.extend(mda_chunks)
print(f"✓ MD&A         : {len(mda_chunks)} chunks")

# ── 3. Chunk 4 quarters earnings ─────────────────────────────────
for _, row in df_edgar.iterrows():
    chunks = chunk_earnings_report(
        raw_text     = row["text"],
        company      = row["company"],
        ticker       = row["ticker"],
        published_at = row["filing_date"],
    )
    all_chunks.extend(chunks)
    print(f"✓ Earnings {row['filing_date']} : {len(chunks)} chunks")

# ── 4. Chunk news articles ───────────────────────────────────────
news_chunks = chunk_news_dataframe(
    df     = news_final,
    ticker = "MSFT",
)
all_chunks.extend(news_chunks)

# ── 5. Convert to dataframe ──────────────────────────────────────
df_chunks = chunks_to_dataframe(all_chunks)

# ── 6. Add section metadata ──────────────────────────────────────
df_chunks["section_type"] = df_chunks["source_type"].map({
    "risk":     "Risk Factors",
    "mda":      "MD&A",
    "earning":  "8-K earnings",
    "earnings": "8-K earnings",
    "news":     "news",
}).fillna("news")

df_chunks["section_weight"] = df_chunks["section_type"].map({
    "MD&A":          1.5,
    "Risk Factors":  1.3,
    "8-K earnings":  1.0,
    "news":          0.8,
}).fillna(1.0)

df_chunks["filing_date"] = df_chunks["published_at"]

# ── 7. Summary ───────────────────────────────────────────────────
print(f"\nFinal df_chunks : {len(df_chunks)} chunks")
print(df_chunks["section_type"].value_counts())

✓ Risk Factors : 52 chunks
✓ MD&A         : 40 chunks
Earnings chunking: 1 report → 4 chunks
✓ Earnings 2026-01-29 : 4 chunks
Earnings chunking: 1 report → 8 chunks
✓ Earnings 2025-10-30 : 8 chunks
Earnings chunking: 1 report → 4 chunks
✓ Earnings 2025-07-31 : 4 chunks
Earnings chunking: 1 report → 4 chunks
✓ Earnings 2025-05-01 : 4 chunks
News chunking    : 34 articles → 126 chunks

Final df_chunks : 238 chunks
section_type
news            126
Risk Factors     52
MD&A             40
8-K earnings     20
Name: count, dtype: int64


In [22]:
df_final = df_chunks

In [23]:
df_final

,company,ticker,source,source_type,published_at,url,chunk_index,chunk_total,text,char_count,token_count,sentiment_label,sentiment_confidence,embedding,section_type,section_weight,filing_date
0,Microsoft,MSFT,SEC EDGAR 10-K,risk,2026-01-28,https://www.sec.gov/cgi-bin/browse-edgar?actio...,0,52,ITEM 1A. RISK FACTORS\n\n Our operations and f...,1584,396,None,None,None,Risk Factors,1.3,2026-01-28
1,Microsoft,MSFT,SEC EDGAR 10-K,risk,2026-01-28,https://www.sec.gov/cgi-bin/browse-edgar?actio...,1,52,Establishing significant scale in the marketpl...,1466,366,None,None,None,Risk Factors,1.3,2026-01-28
2,Microsoft,MSFT,SEC EDGAR 10-K,risk,2026-01-28,https://www.sec.gov/cgi-bin/browse-edgar?actio...,2,52,Users are increasingly turning to these device...,1570,392,None,None,None,Risk Factors,1.3,2026-01-28
3,Microsoft,MSFT,SEC EDGAR 10-K,risk,2026-01-28,https://www.sec.gov/cgi-bin/browse-edgar?actio...,3,52,Competitors’ rules governing their content and...,1522,380,None,None,None,Risk Factors,1.3,2026-01-28
4,Microsoft,MSFT,SEC EDGAR 10-K,risk,2026-01-28,https://www.sec.gov/cgi-bin/browse-edgar?actio...,4,52,•Other competitors develop and offer free appl...,1562,390,None,None,None,Risk Factors,1.3,2026-01-28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233,Apple,MSFT,ZDNet,news,2026-04-21,https://www.zdnet.com/article/diarly-journalin...,1,4,"In fact, I've been using the free version of D...",1595,398,None,None,None,news,0.8,2026-04-21
234,Apple,MSFT,ZDNet,news,2026-04-21,https://www.zdnet.com/article/diarly-journalin...,2,4,Both Moods and Tags make it easier for you to ...,1533,383,None,None,None,news,0.8,2026-04-21
235,Apple,MSFT,ZDNet,news,2026-04-21,https://www.zdnet.com/article/diarly-journalin...,3,4,Any important entry gets a target icon. Screen...,569,142,None,None,None,news,0.8,2026-04-21
236,Apple,MSFT,ZDNet,news,2026-04-20,https://www.zdnet.com/article/why-you-should-w...,0,2,Nina Raemont/ZDNET\n\nFollow ZDNET: Add us as ...,1600,400,None,None,None,news,0.8,2026-04-20


In [24]:
df_final = df_chunks[['ticker', 'source','section_type','text','char_count','filing_date']]
df_chunks['source_type'].value_counts()

source_type
news        126
risk         52
mda          40
earnings     20
Name: count, dtype: int64

In [25]:
df_final['text'][2]

'Users are increasingly turning to these devices to perform functions that in the past were performed by personal computers. Even if many users view these devices as complementary to a personal computer, the prevalence of these devices may make it more difficult to attract application developers to our PC operating system platforms. Competing with operating systems licensed at low or no cost may decrease our PC operating system margins. Popular products or services offered on competing platforms could increase their competitive strength. In addition, some of our devices compete with products made by our original equipment manufacturer (“OEM”) partners, which may affect their commitment to our platform. •Competing platforms have content and application marketplaces with scale and significant installed bases. The variety and utility of content and applications available on a platform are important to device purchasing decisions. Users may incur costs to move data and buy new content and 

In [26]:
df_chunks = df_final

In [27]:
df_chunks

,ticker,source,section_type,text,char_count,filing_date
0,MSFT,SEC EDGAR 10-K,Risk Factors,ITEM 1A. RISK FACTORS\n\n Our operations and f...,1584,2026-01-28
1,MSFT,SEC EDGAR 10-K,Risk Factors,Establishing significant scale in the marketpl...,1466,2026-01-28
2,MSFT,SEC EDGAR 10-K,Risk Factors,Users are increasingly turning to these device...,1570,2026-01-28
3,MSFT,SEC EDGAR 10-K,Risk Factors,Competitors’ rules governing their content and...,1522,2026-01-28
4,MSFT,SEC EDGAR 10-K,Risk Factors,•Other competitors develop and offer free appl...,1562,2026-01-28
...,...,...,...,...,...,...
233,MSFT,ZDNet,news,"In fact, I've been using the free version of D...",1595,2026-04-21
234,MSFT,ZDNet,news,Both Moods and Tags make it easier for you to ...,1533,2026-04-21
235,MSFT,ZDNet,news,Any important entry gets a target icon. Screen...,569,2026-04-21
236,MSFT,ZDNet,news,Nina Raemont/ZDNET\n\nFollow ZDNET: Add us as ...,1600,2026-04-20


In [28]:
# RAG process 

In [29]:
print(df_final.columns.tolist())

['ticker', 'source', 'section_type', 'text', 'char_count', 'filing_date']


#need anthropic api for claude ai 

In [ ]:
# ============================================================
# RAG Pipeline — fixed for df_final columns
# columns: ticker, source, section_type, text, char_count, filing_date
# ============================================================

import os
os.environ["ANTHROPIC_API_KEY"] = 

import anthropic
import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder

# ── CONFIG ───────────────────────────────────────────────────────
EMBED_MODEL  = "all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CHROMA_PATH  = "./chroma_db"
COLLECTION   = "msft_financial"
TOP_K        = 20
TOP_N        = 5

# ── 1. Load models ───────────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)
print("Loading reranking model...")
reranker = CrossEncoder(RERANK_MODEL)
print("✓ Models loaded")

# ── 2. Setup ChromaDB ────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = chroma_client.get_or_create_collection(
    name     = COLLECTION,
    metadata = {"hnsw:space": "cosine"}
)

# ── 3. Index df_final ────────────────────────────────────────────
def index_chunks(df: pd.DataFrame):
    print(f"\nIndexing {len(df)} chunks...")

    texts      = df['text'].tolist()
    embeddings = embedder.encode(texts, show_progress_bar=True)

    collection.upsert(
        ids        = [str(i) for i in df.index],
        embeddings = embeddings.tolist(),
        documents  = texts,
        metadatas  = [
            {
                "section_type": str(row.get('section_type', '')),
                "filing_date":  str(row.get('filing_date', '')),
                "source":       str(row.get('source', '')),
                "ticker":       str(row.get('ticker', 'MSFT')),
                "company":      "Microsoft",
            }
            for _, row in df.iterrows()
        ]
    )
    print(f"✓ {collection.count()} chunks indexed in ChromaDB")

# ── 4. Retrieve ──────────────────────────────────────────────────
def retrieve(query: str, top_k: int = TOP_K) -> list:
    query_vector = embedder.encode([query]).tolist()
    results      = collection.query(
        query_embeddings = query_vector,
        n_results        = top_k,
        include          = ["documents", "metadatas", "distances"]
    )
    candidates = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        candidates.append({
            "text":     doc,
            "metadata": meta,
            "distance": dist
        })
    return candidates

# ── 5. Rerank ────────────────────────────────────────────────────
def rerank(query: str, candidates: list, top_n: int = TOP_N) -> list:
    pairs  = [(query, c['text']) for c in candidates]
    scores = reranker.predict(pairs)
    for i, score in enumerate(scores):
        candidates[i]['rerank_score'] = float(score)
    reranked = sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)
    return reranked[:top_n]

# ── 6. Generate with Claude ──────────────────────────────────────
def generate(query: str, top_chunks: list) -> str:
    context_parts = []
    for i, chunk in enumerate(top_chunks, 1):
        meta = chunk['metadata']
        context_parts.append(
            f"[Source {i} — {meta.get('section_type', 'unknown')} "
            f"from {meta.get('source', 'unknown')} "
            f"dated {meta.get('filing_date', 'unknown')}]\n"
            f"{chunk['text']}"
        )
    context = "\n\n".join(context_parts)

    prompt = f"""You are a financial analyst assistant for Microsoft (MSFT).
Answer the question based ONLY on the provided context from SEC filings and news.
If context does not contain enough information, say so clearly.
Be concise, factual, and cite specific numbers where available.

Context:
{context}

Question: {query}

Answer:"""

    claude   = anthropic.Anthropic()
    response = claude.messages.create(
        model      = "claude-sonnet-4-6",
        max_tokens = 512,
        messages   = [{"role": "user", "content": prompt}]
    )
    return response.content[0].text

# ── 7. Full RAG pipeline ─────────────────────────────────────────
def rag_query(query: str) -> dict:
    print(f"\n{'='*60}")
    print(f"  Query: {query}")
    print(f"{'='*60}")

    candidates = retrieve(query, top_k=TOP_K)
    print(f"  Retrieved : {len(candidates)} candidates")

    top_chunks = rerank(query, candidates, top_n=TOP_N)
    print(f"  Reranked  : top {len(top_chunks)} chunks")
    print(f"\n  Top chunks used:")
    for i, c in enumerate(top_chunks, 1):
        meta = c['metadata']
        print(f"    {i}. [{meta.get('section_type','?')}] "
              f"{meta.get('source','?')} "
              f"({meta.get('filing_date','?')[:10]}) "
              f"score={c['rerank_score']:.3f}")

    answer = generate(query, top_chunks)
    print(f"\n  Answer:")
    print(f"  {answer}")
    print(f"{'='*60}")

    return {
        "query":      query,
        "answer":     answer,
        "top_chunks": top_chunks
    }

# ── RUN ──────────────────────────────────────────────────────────
# Step 1 — index (run once)
index_chunks(df_final)

# Step 2 — test queries
rag_query("What drove Microsoft revenue growth this quarter?")
rag_query("What risks did Microsoft highlight in their filings?")
rag_query("What is happening with Microsoft AI recently?")

#model loaded fine. 

21:53:56  INFO      No device provided, using mps


Loading embedding model...


21:53:57  WARNING   Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
21:53:57  INFO      Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
21:53:59  INFO      No device provided, using mps
21:53:59  INFO      No modules.json found for cross-encoder/ms-marco-MiniLM-L-6-v2, initializing a new CrossEncoder model.


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Retrieved : 20 candidates


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Reranked  : top 5 chunks

  Top chunks used:
    1. [news] The Guardian Business (2026-04-21) score=-0.526
    2. [Risk Factors] SEC EDGAR 10-K (2026-01-28) score=-0.691
    3. [news] TechCrunch (2026-04-18) score=-2.702
    4. [news] CNBC (2026-04-21) score=-3.086
    5. [MD&A] SEC EDGAR 10-K (2026-01-28) score=-3.301

  Answer:
  ## Microsoft AI: Recent Developments

Based on the available context from Microsoft's SEC filings, here is what is happening with Microsoft's AI strategy:

### Strong Financial Performance Driven by AI
Microsoft's AI investments are showing clear revenue results (FY2025 vs FY2024):
- **Microsoft Cloud revenue grew 23% to $168.9 billion**
- **Azure and other cloud services grew 34%**, the strongest growth metric reported
- **Dynamics 365 grew 19%**, also largely AI-driven
- **Search and news advertising** (benefiting from AI integration) grew **20%**

### Strategic Vision
Per the 10-K (filed January 28, 2026), Microsoft envisions *"a future in which AI oper

{'query': 'What is happening with Microsoft AI recently?',
 'answer': '## Microsoft AI: Recent Developments\n\nBased on the available context from Microsoft\'s SEC filings, here is what is happening with Microsoft\'s AI strategy:\n\n### Strong Financial Performance Driven by AI\nMicrosoft\'s AI investments are showing clear revenue results (FY2025 vs FY2024):\n- **Microsoft Cloud revenue grew 23% to $168.9 billion**\n- **Azure and other cloud services grew 34%**, the strongest growth metric reported\n- **Dynamics 365 grew 19%**, also largely AI-driven\n- **Search and news advertising** (benefiting from AI integration) grew **20%**\n\n### Strategic Vision\nPer the 10-K (filed January 28, 2026), Microsoft envisions *"a future in which AI operating in our devices, applications, and the cloud helps our customers be more productive in their work and personal lives."*\n\n### Acknowledged Risks\nMicrosoft explicitly flags AI-related risks in its SEC filings, including:\n- Flawed algorithms or

In [32]:
# skip re-indexing — already done
# just run the queries

result1 = rag_query("What drove Apple revenue growth this quarter?")
print("\n")
result2 = rag_query("What risks did Apple highlight in their filings?")
print("\n")
result3 = rag_query("What is happening with Apple AI recently?")
result4 = rag_query("What is the latest news about Apple?")
result5 = rag_query("What did analysts say about Apple recently?")
result6 = rag_query("What Apple products were released recently?")
result7 = rag_query("What leadership changes happened at Apple?")
result8 = rag_query("What security issues did Apple have recently?")


  Query: What drove Apple revenue growth this quarter?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Retrieved : 20 candidates


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Reranked  : top 5 chunks

  Top chunks used:
    1. [8-K earnings] SEC EDGAR 8-K (2025-05-01) score=5.898
    2. [8-K earnings] SEC EDGAR 8-K (2025-07-31) score=5.264
    3. [8-K earnings] SEC EDGAR 8-K (2026-01-29) score=4.679
    4. [8-K earnings] SEC EDGAR 8-K (2025-10-30) score=4.212
    5. [news] Wired (2026-04-21) score=1.089

  Answer:
  Based on the provided context, I need to flag an important issue: **this assistant is configured as a Microsoft (MSFT) financial analyst**, and all provided sources relate to **Apple (AAPL)**, not Microsoft.

That said, answering based solely on the provided context:

---

**Most Recent Quarter (Q1 FY2026, ended December 27, 2025):**

Apple's **16% year-over-year revenue growth to $143.8 billion** (an all-time record) was driven by:

- **iPhone**: Best-ever quarter with "unprecedented demand" and all-time records across every geographic segment
- **Services**: All-time revenue record of **~$30 billion**, up **14% year-over-year**, surpassing M

...

  Reranked  : top 5 chunks

  Top chunks used:
    1. [news] Fortune (2026-04-21) score=2.467
    2. [news] Fortune (2026-04-21) score=0.822
    3. [news] The Guardian Tech (2026-04-21) score=0.198
    4. [news] TechCrunch (2026-04-21) score=-0.285
    5. [news] TechCrunch (2026-04-21) score=-0.285

  Answer:
  ## Apple Leadership Changes

Based on the provided sources, Apple announced a significant CEO transition:

**Key Changes:**
- **Tim Cook** will step down as CEO on **September 1, 2026** after nearly 15 years in the role, transitioning to the position of **Executive Chairman** of Apple's board of directors
- **John Ternus**, currently Senior Vice President of Hardware Engineering, will succeed Cook as **CEO effective September 1, 2026**

**About the Incoming CEO:**
- Ternus, age 50, joined Apple in 2001 and has spent almost his entire career there
- He became VP of Hardware Engineering in 2013 and head of the department in 2021
- He oversaw engineering for iPhone, iPad, Mac, Appl

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Retrieved : 20 candidates


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Reranked  : top 5 chunks

  Top chunks used:
    1. [news] TechCrunch (2026-04-18) score=-0.228
    2. [news] The Guardian Tech (2026-04-21) score=-0.911
    3. [news] The Guardian Tech (2026-04-21) score=-1.504
    4. [news] CNBC (2026-04-21) score=-2.083
    5. [news] The Guardian Tech (2026-04-21) score=-3.058

  Answer:
  ## Recent Apple Security Issues

Based on the provided context, Apple has faced several notable security concerns:

### App Store Security Failures
- **Malicious crypto app**: A clone of **Ledger Live** slipped through App Store review and **drained $9.5 million in crypto** from victims' accounts *(TechCrunch, April 2026)*
- **Freecash rewards app**: Apple allowed this app to climb to the **top five in the App Store charts for months** before pulling it for rules violations — highlighting gaps in proactive monitoring *(TechCrunch, April 2026)*

### Law Enforcement & Privacy Tensions
- Despite Apple's strong public stance on privacy (e.g., refusing FBI access to 

In [ ]:


# print full text of each chunk without truncation
for i, chunk in enumerate(result['top_chunks'], 1):
    print(f"\n{'='*60}")
    print(f"CHUNK {i} FULL TEXT")
    print(f"{'='*60}")
    print(chunk['text'])
# ── retrieve by section ──────────────────────────────────────────
def retrieve_by_section(query: str, section_type: str, top_k: int = 10) -> list:
    query_vector = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings = query_vector,
        n_results        = top_k,
        where            = {"section_type": section_type},
        include          = ["documents", "metadatas", "distances"]
    )
    candidates = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        candidates.append({
            "text":     doc,
            "metadata": meta,
            "distance": dist
        })
    return candidates

# ── hybrid RAG query ─────────────────────────────────────────────
def rag_query_hybrid(query: str) -> dict:
    print(f"\n{'='*60}")
    print(f"  Query (hybrid): {query}")
    print(f"{'='*60}")

    # retrieve from each section separately
    news_candidates     = retrieve_by_section(query, "news",         top_k=8)
    earnings_candidates = retrieve_by_section(query, "8-K earnings", top_k=6)
    risk_candidates     = retrieve_by_section(query, "Risk Factors", top_k=4)
    mda_candidates      = retrieve_by_section(query, "MD&A",         top_k=4)

    all_candidates = (news_candidates + earnings_candidates +
                     risk_candidates + mda_candidates)

    top_chunks = rerank(query, all_candidates, top_n=5)

    print(f"\n  Top chunks used:")
    for i, c in enumerate(top_chunks, 1):
        meta = c['metadata']
        print(f"\n  ── Chunk {i} ──────────────────────────────────")
        print(f"  Section : {meta.get('section_type','?')}")
        print(f"  Source  : {meta.get('source','?')}")
        print(f"  Date    : {meta.get('filing_date','?')[:10]}")
        print(f"  Score   : {c['rerank_score']:.3f}")
        print(f"  Text    : {c['text']}")

    answer = generate(query, top_chunks)
    print(f"\n  Answer:")
    print(f"  {answer}")

    return {
        "query":      query,
        "answer":     answer,
        "top_chunks": top_chunks
    }

# ── test ─────────────────────────────────────────────────────────

In [ ]:
rag_query_hybrid("What is Apple doing with employee monitoring?")
rag_query_hybrid("What leadership changes happened at Apple?")
rag_query_hybrid("What security issues did Apple have recently?")
rag_query_hybrid("What drove Apple revenue growth this quarter?")


  Query (hybrid): What is Apple doing with employee monitoring?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


  Top chunks used:

  ── Chunk 1 ──────────────────────────────────
  Section : 8-K earnings
  Source  : SEC EDGAR 8-K
  Date    : 2025-05-01
  Score   : 5.898
  Text    : Apple reports second quarter results
Services revenue reaches new all-time high
EPS sets March quarter record
CUPERTINO, CALIFORNIA — Apple® today announced financial results for its fiscal 2025 second quarter ended March 29, 2025. The Company posted quarterly revenue of $95.4 billion, up 5 percent year over year, and quarterly diluted earnings per share of $1.65, up 8 percent year over year. “Today Apple is reporting strong quarterly results, including double-digit growth in Services,” said Tim Cook, Apple’s CEO. “We were happy to welcome iPhone 16e to our lineup, and to introduce powerful new Macs and iPads that take advantage of the extraordinary capabilities of Apple silicon. And we were proud to announce that we’ve cut our carbon emissions by 60 percent over the past decade.”
“Our March quarter business perform

{'query': 'What drove Apple revenue growth this quarter?',
 'answer': '## Apple Revenue Growth Drivers — Q1 FY2026 (Quarter Ended December 27, 2025)\n\nBased on the most recent quarterly earnings available (Source 3 — 8-K dated 2026-01-29), Apple posted **$143.8 billion in revenue, up 16% year over year**, driven by the following key factors:\n\n### Primary Growth Drivers:\n1. **iPhone** — Achieved its **best-ever quarter**, with all-time records across **every geographic segment**, driven by "unprecedented demand"\n2. **Services** — Reached an **all-time revenue record**, up **14% year over year**, hitting approximately **$30 billion** (per Source 5)\n\n### Additional Highlights:\n- **EPS** of **$2.84**, up **19% YoY**, setting a new all-time EPS record\n- **Operating cash flow** of nearly **$54 billion**, enabling ~$32 billion returned to shareholders\n- **Installed base** surpassed **2.5 billion active devices**, reflecting strong customer loyalty\n\n### Context:\nCEO Tim Cook descr